In [4]:
import torch
import gpytorch
from gpytorch.likelihoods import MultitaskGaussianLikelihood
from botorch.utils.probability.truncated_multivariate_normal import TruncatedMultivariateNormal
from torch.distributions import constraints

class MultivariateTruncatedNormalLikelihood(MultitaskGaussianLikelihood):
    """
    A custom likelihood for multivariate truncated normal distributions that extends
    gpytorch's MultitaskGaussianLikelihood and uses botorch's TruncatedMultivariateNormal.
    """
   
    def __init__(self, num_tasks, truncation_lower=None, truncation_upper=None, **kwargs):
        """
        Initialize the likelihood with truncation bounds.
       
        Args:
            num_tasks: Number of tasks/outputs
            truncation_lower: Lower bounds for truncation (None or tensor of shape [num_tasks])
            truncation_upper: Upper bounds for truncation (None or tensor of shape [num_tasks])
            **kwargs: Additional arguments passed to parent class
        """
        super().__init__(num_tasks=num_tasks, **kwargs)
       
        # Store truncation bounds
        self.truncation_lower = truncation_lower
        self.truncation_upper = truncation_upper
       
        # Validate truncation bounds
        if truncation_lower is not None and truncation_upper is not None:
            if not (truncation_lower <= truncation_upper).all():
                raise ValueError("Lower bounds must be less than or equal to upper bounds")
   
    def forward(self, function_samples, targets=None, **kwargs):
        """
        Forward pass to compute the likelihood.
       
        Args:
            function_samples: Tensor of function samples [batch_size, num_tasks]
            targets: Target values [batch_size, num_tasks]
           
        Returns:
            Distribution object representing the likelihood
        """
        # Get the base Gaussian distribution from parent class
        base_dist = super().forward(function_samples, targets=targets, **kwargs)
       
        # Extract mean and covariance
        mean = base_dist.mean
        cov = base_dist.covariance_matrix
       
        # Create truncated multivariate normal distribution
        if self.truncation_lower is not None and self.truncation_upper is not None:
            # Use botorch's TruncatedMultivariateNormal
            truncated_dist = TruncatedMultivariateNormal(
                mean=mean,
                covariance_matrix=cov,
                lower_bounds=self.truncation_lower,
                upper_bounds=self.truncation_upper
            )
        else:
            # Fall back to regular multivariate normal if no truncation
            truncated_dist = base_dist
           
        return truncated_dist
   
    def expected_log_prob(self, observations, function_dist, *args, **kwargs):
        """
        Compute the expected log probability of observations under the likelihood.
       
        Args:
            observations: Observed values [batch_size, num_tasks]
            function_dist: Distribution of function values
           
        Returns:
            Expected log probability
        """
        # Get the truncated distribution
        mean = function_dist.mean
        cov = function_dist.covariance_matrix
       
        if self.truncation_lower is not None and self.truncation_upper is not None:
            truncated_dist = TruncatedMultivariateNormal(
                mean=mean,
                covariance_matrix=cov,
                lower_bounds=self.truncation_lower,
                upper_bounds=self.truncation_upper
            )
        else:
            truncated_dist = function_dist
           
        # Compute log probability
        log_prob = truncated_dist.log_prob(observations)
        return log_prob

# Example usage
if __name__ == "__main__":
    # Example: Create a likelihood with truncation bounds
    num_tasks = 3
    lower_bounds = torch.tensor([-1.0, -2.0, -1.5])
    upper_bounds = torch.tensor([2.0, 3.0, 2.5])
   
    likelihood = MultivariateTruncatedNormalLikelihood(
        num_tasks=num_tasks,
        truncation_lower=lower_bounds,
        truncation_upper=upper_bounds
    )
   
    print("Custom Multivariate Truncated Normal Likelihood created successfully!")
    print(f"Number of tasks: {num_tasks}")
    print(f"Lower bounds: {lower_bounds}")
    print(f"Upper bounds: {upper_bounds}")

# In your model definition
class MyModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super().__init__(train_x, train_y, likelihood)
        # ... rest of your model
       
# Create the likelihood
likelihood = MultivariateTruncatedNormalLikelihood(
    num_tasks=3,
    truncation_lower=lower_bounds,
    truncation_upper=upper_bounds,
)

Custom Multivariate Truncated Normal Likelihood created successfully!
Number of tasks: 3
Lower bounds: tensor([-1.0000, -2.0000, -1.5000])
Upper bounds: tensor([2.0000, 3.0000, 2.5000])
